# Visual-language assistant with Ministral-3 and OpenVINO

Ministral-3 (Ministral-3-3B-Instruct-2512) is a lightweight, state-of-the-art multimodal model from [Mistral AI](https://mistral.ai/), combining a 3.4B parameter language model with a 0.4B parameter vision encoder based on the Pixtral architecture. It is designed for efficient visual-language understanding tasks.

**Key Features of Ministral-3:**
* **Multimodal Understanding**: Combines text and vision capabilities in a compact 3B parameter model, enabling image understanding and visual question answering.
* **Long Context Support**: Supports up to 262,144 tokens with YaRN RoPE scaling for extended context processing.
* **Efficient Architecture**: Uses Grouped Query Attention (32 attention heads with 8 KV heads) for memory-efficient inference.
* **Pixtral Vision Encoder**: Employs a PixtralVisionModel with patch-based image processing and multi-modal projection for seamless vision-language integration.

More details about the model can be found in the [model card](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512) and the [Mistral AI documentation](https://docs.mistral.ai/).

In this tutorial we consider how to convert and optimize Ministral-3 model for creating a multimodal chatbot using [Optimum Intel](https://github.com/huggingface/optimum-intel). Additionally, we demonstrate how to apply model optimization techniques like weights compression using [NNCF](https://github.com/openvinotoolkit/nncf).

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
    - [Compress model weights to 4-bit](#Compress-model-weights-to-4-bit)
- [Run model inference](#Run-model-inference)
    - [Select inference device](#Select-inference-device)
    - [Load OpenVINO model](#Load-OpenVINO-model)
    - [Run conversation with image](#Run-conversation-with-image)
- [Interactive Demo](#Interactive-Demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/ministral-3/ministral-3.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install required packages and setup helper functions.

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q -U "nncf>=2.15.0" "torch==2.8" "torchvision==0.23.0" "transformers>=4.50.0" "peft>=0.15.0" "Pillow" "gradio>=4.36,<6" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q -U "openvino>=2025.0.0" "openvino-tokenizers>=2025.0.0"
# Install optimum-intel from PR #1659 which adds Mistral3 (ministral-3b) export and inference support
%pip install -q --upgrade-strategy eager "git+https://github.com/huggingface/optimum-intel.git@refs/pull/1659/head#egg=optimum-intel[openvino,nncf]" --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("ministral-3.ipynb")

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

Ministral-3 is a PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.compile_model`.

For convenience, we will use OpenVINO integration with HuggingFace Optimum. [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for, if not specified, the task will be auto-inferred based on the model. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`. For int8 and int4, [nncf](https://github.com/openvinotoolkit/nncf) will be used for weight compression. More details about model export provided in [Optimum Intel documentation](https://huggingface.co/docs/optimum/intel/openvino/export#export-your-model).


### Compress model weights to 4-bit
[back to top ⬆️](#Table-of-contents:)

For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf).

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

`nncf.compress_weights` function can be used for performing weights compression. The function accepts an OpenVINO model and other compression parameters. Compared to INT8 compression, INT4 compression improves performance even more, but introduces a minor drop in prediction quality.

More details about weights compression, can be found in [OpenVINO documentation](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html).
</details>

In [ ]:
from cmd_helper import optimum_cli

pt_model_id = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
model_dir = Path("Ministral-3-3B-Instruct-2512-BF16")

if not (model_dir / "INT4").exists():
    optimum_cli(pt_model_id, model_dir / "INT4", additional_args={"weight-format": "int4", "task": "image-text-to-text"})

## Run model inference
[back to top ⬆️](#Table-of-contents:)

OpenVINO integration with Optimum Intel provides ready-to-use API for model inference that can be used for smooth integration with transformers-based solutions. For loading model, we will use `OVModelForVisualCausalLM` class that has compatible API with Transformers models and provides the following interface for interaction:

* `from_pretrained` - for loading model from directory.
* `generate` - for running model inference.
* `preprocess_inputs` - for preparing model inputs.

In [ ]:
from optimum.intel.openvino import OVModelForVisualCausalLM
from transformers import AutoProcessor, TextStreamer

### Select inference device
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

device

### Load OpenVINO model
[back to top ⬆️](#Table-of-contents:)

For model loading we should provide path to model directory and inference device.

In [ ]:
model_export_dir = model_dir / "INT4"

model = OVModelForVisualCausalLM.from_pretrained(model_export_dir, device=device.value)
processor = AutoProcessor.from_pretrained(model_export_dir)

### Run conversation with image
[back to top ⬆️](#Table-of-contents:)

Now, when we have model and processor loaded, we can run model inference.

For preparing input data, we use `preprocess_inputs` method that accepts text and image, and returns model-ready inputs. Additionally, we use `TextStreamer` for streaming token-by-token output.

In [ ]:
from PIL import Image
from io import BytesIO

MAX_IMAGE_SIZE = 512


def load_image(image_file):
    if isinstance(image_file, str) and (image_file.startswith("http") or image_file.startswith("https")):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_file).convert("RGB")
    # Resize large images to keep patch count manageable
    if max(image.size) > MAX_IMAGE_SIZE:
        image.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE))
    return image


image_url = "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/d5fbbd1a-d484-415c-88cb-9986625b7b11"
image_file = Path("cat.png")
text_message = "What is unusual on this image?"

if not image_file.exists():
    image = load_image(image_url)
    image.save(image_file)
else:
    image = load_image(image_file)

inputs = model.preprocess_inputs(text=text_message, image=image, processor=processor)

In [ ]:
print(f"Question:\n{text_message}")
display(image)
print("Answer:")
model.generate(**inputs, do_sample=False, max_new_tokens=128, streamer=TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True));

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/ministral-3/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, processor)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')